# 00 · Mapillary Light Fetching — Other Cities

Fetches Mapillary street-light detections (`object--street-light`) via the Mapillary Graph API v4 for five neighbourhoods:

| Neighbourhood | City |
|---|---|
| Eixample | Barcelona |
| Raval | Barcelona |
| Belleville | Paris |
| Trastevere | Rome |
| Salamanca | Madrid |

Output: one CSV per neighbourhood saved to `data/mapillary/` with columns `id, lon, lat, first_seen_at, last_seen_at, object_type`.

## 0 · Imports & config

In [9]:
import os
import time
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import shape, Point

MAPILLARY_TOKEN_NAME = 'MAPILLARY_TOKEN'


def _parse_env_value(raw_value: str) -> str:
    value = raw_value.strip()
    if not value:
        return value

    if value[0] in {'"', "'"}:
        quote = value[0]
        end = 1
        while end < len(value):
            if value[end] == quote and value[end - 1] != '\\':
                return value[1:end]
            end += 1
        return value[1:]

    if ' #' in value:
        value = value.split(' #', 1)[0].strip()
    elif '#' in value:
        value = value.split('#', 1)[0].strip()

    return value


def _read_env_file_value(env_path: Path, key: str) -> str | None:
    if not env_path.exists():
        return None

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue
        candidate_key, candidate_value = line.split('=', 1)
        if candidate_key.strip() != key:
            continue
        return _parse_env_value(candidate_value)

    return None


def _load_mapillary_token() -> str | None:
    token = os.environ.get(MAPILLARY_TOKEN_NAME)
    if token:
        return token

    for base_dir in (Path.cwd(), Path.cwd().parent):
        token = _read_env_file_value(base_dir / '.env', MAPILLARY_TOKEN_NAME)
        if token:
            return token

    return None


MAPILLARY_TOKEN = _load_mapillary_token()
assert MAPILLARY_TOKEN, (
    'Set MAPILLARY_TOKEN in your environment or a .env file next to the notebook or repo root'
)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('data/mapillary')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── CRS ───────────────────────────────────────────────────────────────────────
CRS_GEO        = 'EPSG:4326'
CRS_PROJECTED  = 'EPSG:3857'   # web mercator; change to a local UTM if preferred

# ── API settings ──────────────────────────────────────────────────────────────
BASE_URL       = 'https://graph.mapillary.com/map_features'
OBJECT_TYPE    = 'object--street-light'
PAGE_LIMIT     = 2000   # max per request (Mapillary hard limit)
SLEEP_BETWEEN  = 0.5    # seconds between paginated requests (be polite)

## 1 · Neighbourhood bounding boxes

Bounding boxes in `[min_lon, min_lat, max_lon, max_lat]` (EPSG:4326).  
Tighten or expand these if you want to cover a different extent.

In [10]:
NEIGHBOURHOODS = {
    # Barcelona
    'eixample_barcelona':  [2.1467, 41.3830, 2.1810, 41.4020],
    'raval_barcelona':     [2.1620, 41.3760, 2.1750, 41.3870],
    # Paris
    'belleville_paris':    [2.3720, 48.8650, 2.3970, 48.8780],
    # Rome
    'trastevere_rome':     [12.4650, 41.8820, 12.4810, 41.8940],
    # Madrid
    'salamanca_madrid':    [-3.6820, 40.4230, -3.6640, 40.4370],
}

print('Neighbourhoods configured:')
for name, bbox in NEIGHBOURHOODS.items():
    print(f'  {name:30s}  bbox={bbox}')

Neighbourhoods configured:
  eixample_barcelona              bbox=[2.1467, 41.383, 2.181, 41.402]
  raval_barcelona                 bbox=[2.162, 41.376, 2.175, 41.387]
  belleville_paris                bbox=[2.372, 48.865, 2.397, 48.878]
  trastevere_rome                 bbox=[12.465, 41.882, 12.481, 41.894]
  salamanca_madrid                bbox=[-3.682, 40.423, -3.664, 40.437]


## 2 · Fetch helpers

In [11]:
def fetch_mapillary_lights_bbox(bbox: list[float], token: str) -> pd.DataFrame:
    """
    Fetch all Mapillary street-light map features inside *bbox*.
    Handles cursor-based pagination automatically.

    Parameters
    ----------
    bbox  : [min_lon, min_lat, max_lon, max_lat]
    token : Mapillary Graph API access token

    Returns
    -------
    pd.DataFrame with columns: id, lon, lat, first_seen_at, last_seen_at, object_type
    """
    records = []
    params = {
        'access_token': token,
        'fields':       'id,geometry,first_seen_at,last_seen_at,object_type',
        'object_type':  OBJECT_TYPE,
        'bbox':         ','.join(map(str, bbox)),
        'limit':        PAGE_LIMIT,
    }

    url = BASE_URL
    page = 0
    while url:
        resp = requests.get(url, params=params if page == 0 else None)
        resp.raise_for_status()
        payload = resp.json()

        for feat in payload.get('data', []):
            geom = feat.get('geometry', {})
            coords = geom.get('coordinates', [None, None])
            records.append({
                'id':            feat.get('id'),
                'lon':           coords[0],
                'lat':           coords[1],
                'first_seen_at': feat.get('first_seen_at'),
                'last_seen_at':  feat.get('last_seen_at'),
                'object_type':   feat.get('object_type'),
            })

        # Pagination cursor
        paging = payload.get('paging', {})
        next_url = paging.get('next')
        url = next_url  # None → loop ends
        params = None   # cursor is already embedded in next_url
        page += 1

        if next_url:
            time.sleep(SLEEP_BETWEEN)

    return pd.DataFrame(records)


def load_mapillary_lights(csv_path: Path) -> gpd.GeoDataFrame | None:
    """
    Load a previously saved Mapillary CSV and return a projected GeoDataFrame.
    Mirrors the helper from the London notebook; accepts lat/lon or latitude/longitude.
    """
    if not csv_path.exists():
        print(f'  [WARNING] Mapillary CSV not found at {csv_path}.')
        print('  lamp_count will be set to NaN for all segments.')
        return None

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.lower().str.strip()

    lat_col = next((c for c in df.columns if c.startswith('lat')), None)
    lon_col = next((c for c in df.columns if c.startswith('lon')), None)
    if lat_col is None or lon_col is None:
        raise ValueError(
            f'Could not find lat/lon columns in {csv_path}. Found: {list(df.columns)}'
        )

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs=CRS_GEO,
    ).to_crs(CRS_PROJECTED)

    print(f'  Loaded {len(gdf):,} Mapillary light points from {csv_path.name}.')
    return gdf

## 3 · Fetch & save all neighbourhoods

In [12]:
results = {}

for name, bbox in NEIGHBOURHOODS.items():
    out_path = DATA_DIR / f'mapillary_lights_{name}.csv'

    if out_path.exists():
        print(f'[SKIP] {name} — already fetched ({out_path})')
        results[name] = pd.read_csv(out_path)
        continue

    print(f'[FETCH] {name} …', end=' ', flush=True)
    try:
        df = fetch_mapillary_lights_bbox(bbox, MAPILLARY_TOKEN)
        df.to_csv(out_path, index=False)
        results[name] = df
        print(f'{len(df):,} lights → {out_path}')
    except requests.HTTPError as e:
        print(f'ERROR: {e}')
        results[name] = None

    time.sleep(SLEEP_BETWEEN)

[FETCH] eixample_barcelona … 1,852 lights → data\mapillary\mapillary_lights_eixample_barcelona.csv
[FETCH] raval_barcelona … 1,763 lights → data\mapillary\mapillary_lights_raval_barcelona.csv
[FETCH] belleville_paris … 1,210 lights → data\mapillary\mapillary_lights_belleville_paris.csv
[FETCH] trastevere_rome … 1,843 lights → data\mapillary\mapillary_lights_trastevere_rome.csv
[FETCH] salamanca_madrid … 1,819 lights → data\mapillary\mapillary_lights_salamanca_madrid.csv


## 4 · Summary

In [13]:
summary = [
    {
        'neighbourhood': name,
        'n_lights':      len(df) if df is not None else 'ERROR',
        'csv':           str(DATA_DIR / f'mapillary_lights_{name}.csv'),
    }
    for name, df in results.items()
]
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

     neighbourhood  n_lights                                                    csv
eixample_barcelona      1852 data\mapillary\mapillary_lights_eixample_barcelona.csv
   raval_barcelona      1763    data\mapillary\mapillary_lights_raval_barcelona.csv
  belleville_paris      1210   data\mapillary\mapillary_lights_belleville_paris.csv
   trastevere_rome      1843    data\mapillary\mapillary_lights_trastevere_rome.csv
  salamanca_madrid      1819   data\mapillary\mapillary_lights_salamanca_madrid.csv


## 5 · Quick sanity check — load back as GeoDataFrames

In [14]:
gdfs = {}
for name in NEIGHBOURHOODS:
    csv_path = DATA_DIR / f'mapillary_lights_{name}.csv'
    print(f'Loading {name}:')
    gdfs[name] = load_mapillary_lights(csv_path)

Loading eixample_barcelona:
  Loaded 1,852 Mapillary light points from mapillary_lights_eixample_barcelona.csv.
Loading raval_barcelona:
  Loaded 1,763 Mapillary light points from mapillary_lights_raval_barcelona.csv.
Loading belleville_paris:
  Loaded 1,210 Mapillary light points from mapillary_lights_belleville_paris.csv.
Loading trastevere_rome:
  Loaded 1,843 Mapillary light points from mapillary_lights_trastevere_rome.csv.
Loading salamanca_madrid:
  Loaded 1,819 Mapillary light points from mapillary_lights_salamanca_madrid.csv.


## 6 · Quick visual check (optional)

Plots detected lights for each neighbourhood. Requires `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(gdfs), figsize=(5 * len(gdfs), 5))
if len(gdfs) == 1:
    axes = [axes]

for ax, (name, gdf) in zip(axes, gdfs.items()):
    if gdf is None or gdf.empty:
        ax.set_title(f'{name}\n(no data)')
        continue
    gdf.plot(ax=ax, markersize=2, color='orange', alpha=0.6)
    ax.set_title(f'{name}\n({len(gdf):,} lights)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(DATA_DIR / 'mapillary_lights_overview.png', dpi=150)
plt.show()
print('Saved overview plot to', DATA_DIR / 'mapillary_lights_overview.png')